# Forecast with known-future exogenous variables

Use **`df`** for history (`unique_id`, `ds`, `y`, and exog columns in the past) and **`X_df`** for
known-future exog over the next `h` steps (no `y`). See [Exogenous variables](../exogenous-variables.md)
for the full API.

**Dataset:** panel built from fev-bench `epf_*` markets (BE, DE, FR, NP, PJM) with two covariates
per market, unified as `ex_1` and `ex_2`.

**Models:** `Tafsut` uses `exog_strategy="auto"` (linear `XReg` fallback) and **Chronos-2** uses native
exog. Both support **`level`** with exog.

## Import libraries

In [ ]:
import pandas as pd

from foundationforecast import FoundationForecast

## Load the dataset

- **`train.parquet`:** history with `y`, `ex_1`, and `ex_2`
- **`test.parquet`:** 24-hour horizon with exog (and `y` for evaluation)

For cross-validation we concatenate train and test so the last `h` steps match the fev-bench holdout.
`X_df` carries horizon exog only.

In [ ]:
TRAIN_URL = "https://timecopilot.s3.amazonaws.com/public/data/electricity_price/train.parquet"
TEST_URL = "https://timecopilot.s3.amazonaws.com/public/data/electricity_price/test.parquet"

train = pd.read_parquet(TRAIN_URL)
test = pd.read_parquet(TEST_URL)
df = pd.concat([train, test], ignore_index=True).sort_values(["unique_id", "ds"])

df.head()


In [ ]:
X_df.head()

In [ ]:
y_test_df.head()

## Plot the data

In [ ]:
FoundationForecast.plot(df, max_insample_length=24 * 7)

## Import the models

In [ ]:
from foundationforecast.models import Chronos, Tafsut

## Create a FoundationForecast

In [ ]:
models = [
    Tafsut(alias="Tafsut", exog_strategy="auto"), # default is "auto"
    Chronos(repo_id="amazon/chronos-2", alias="Chronos-2"),
]

ff = FoundationForecast(models=models)

## Generate forecast

In [ ]:
level = [80, 90]
h = 24
exog = ["ex_1", "ex_2"]

cv_df = ff.cross_validation(
    df=df,
    h=h,
    freq="h",
    level=level,
    n_windows=1,
    futr_exog_list=exog,
)


In [ ]:
ff.plot(
    df.query("unique_id == 'DE'"),
    cv_df.query("unique_id == 'DE'").drop(columns=["cutoff", "y"]),
    level=[80],
)

In [ ]:
cv_df.head()

## Evaluation

In [ ]:
from functools import partial

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, scaled_crps

In [ ]:
eval_df = evaluate(
    cv_df.drop(columns=["cutoff"]),
    train_df=train,
    metrics=[partial(mase, seasonality=24), scaled_crps],
    level=level,
)
eval_df.groupby("metric").mean(numeric_only=True).T.round(3)
